# v3a — Three-tier fraud triage

Transactions are scored by the ensemble and routed to one of three outcomes:
- **auto-fraud** — block immediately (score >= `fraud_threshold`, targeting >=70% precision)
- **manual review** — uncertain zone; surfaced to analysts as a review queue
- **auto-legit** — pass through (score < `legit_threshold`; at most 10% of fraud falls here)

In [1]:
pip install xgboost scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, classification_report,
)
from sklearn.model_selection import train_test_split

SEED = 0
np.random.seed(SEED)

In [3]:
dataset1 = pd.read_csv("../00_reference/RTF_2.csv")   # train + val
dataset2 = pd.read_csv("../00_reference/RTF_1.csv")   # held-out test (out-of-distribution)

In [4]:
dataset1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1201 entries, 0 to 1200
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           1201 non-null   object 
 1   batch_id           1001 non-null   float64
 2   card_match         1201 non-null   int64  
 3   cvv                1201 non-null   int64  
 4   is_card_payment    1201 non-null   bool   
 5   is_cash_payment    1201 non-null   bool   
 6   is_delivery        1201 non-null   bool   
 7   is_dine_in         1201 non-null   bool   
 8   is_takeaway        1201 non-null   bool   
 9   is_wallet_payment  1201 non-null   bool   
 10  first_trained_at   0 non-null      float64
 11  last_trained_at    0 non-null      float64
 12  is_fraud           1201 non-null   bool   
dtypes: bool(7), float64(3), int64(2), object(1)
memory usage: 64.6+ KB


In [5]:
dataset2.head()

,order_id,batch_id,card_match,cvv,is_card_payment,is_cash_payment,is_delivery,is_dine_in,is_takeaway,is_wallet_payment,first_trained_at,last_trained_at,is_fraud
0,ORD-260527-926835,3.0,1,414,True,False,True,False,False,False,NaN,NaN,True
1,ORD-260527-960506,3.0,1,761,True,False,True,False,False,False,NaN,NaN,True
2,ORD-260527-400985,3.0,1,204,True,False,True,False,False,False,NaN,NaN,True
3,ORD-260527-247461,3.0,1,867,True,False,True,False,False,False,NaN,NaN,True
4,ORD-260527-165011,3.0,1,362,True,False,True,False,False,False,NaN,NaN,True


In [6]:
EXPECTED_COLS = {
    'order_id', 'batch_id', 'card_match', 'cvv',
    'is_card_payment', 'is_cash_payment', 'is_delivery',
    'is_dine_in', 'is_takeaway', 'is_wallet_payment',
    'first_trained_at', 'last_trained_at', 'is_fraud',
}

for name, df in [("RTF_2 (train/val)", dataset1), ("RTF_1 (test)", dataset2)]:
    missing = EXPECTED_COLS - set(df.columns)
    extra   = set(df.columns) - EXPECTED_COLS
    if missing:
        raise ValueError(f"{name}: missing expected columns: {missing}")
    if extra:
        print(f"WARNING {name}: unexpected columns: {extra}")
    if df['is_fraud'].isna().any():
        raise ValueError(f"{name}: 'is_fraud' contains nulls")
    for col in ['first_trained_at', 'last_trained_at']:
        if df[col].notna().any():
            print(f"WARNING {name}.{col}: has non-null values -- review before treating as a feature")

fraud_train = dataset1['is_fraud'].mean()
fraud_test  = dataset2['is_fraud'].mean()
print(f"RTF_2 (train/val) fraud rate: {fraud_train:.2%}  ({int(dataset1['is_fraud'].sum())}/{len(dataset1)})")
print(f"RTF_1 (test)      fraud rate: {fraud_test:.2%}  ({int(dataset2['is_fraud'].sum())}/{len(dataset2)})")
if abs(fraud_train - fraud_test) > 0.10:
    print(f"WARNING: class distribution shift -- train/val={fraud_train:.0%} fraud, test={fraud_test:.0%} fraud")

RTF_2 (train/val) fraud rate: 16.65%  (200/1201)
RTF_1 (test)      fraud rate: 83.40%  (1000/1199)


## Feature configuration

`order_id` is kept as metadata for the review table rather than dropped.
`batch_id` is excluded: in RTF_1, batch 3 is 100% fraud and batch 4 is 100% legit — including it would let the model memorise batch IDs instead of learning transaction-level patterns.

Three interaction features are added from the raw (0/1) boolean values before normalisation:
- `card_pay_mismatch` — card payment where the card number doesn't match
- `delivery_mismatch` — delivery order where the card number doesn't match
- `card_pay_delivery` — card payment for a delivery order

In [7]:
BOOL_COLS = [
    'is_card_payment', 'is_cash_payment', 'is_delivery',
    'is_dine_in', 'is_takeaway', 'is_wallet_payment',
]
NUM_COLS  = ['card_match', 'cvv'] + BOOL_COLS
TARGET    = 'is_fraud'
DROP_COLS = ['batch_id', 'first_trained_at', 'last_trained_at']

# Interaction features derived from raw (0/1) values before normalisation
INTERACTION_COLS = ['card_pay_mismatch', 'delivery_mismatch', 'card_pay_delivery']
ALL_FEATURE_COLS = NUM_COLS + INTERACTION_COLS

def add_interactions(df):
    df = df.copy()
    df['card_pay_mismatch'] = df['is_card_payment'] * (1.0 - df['card_match'])
    df['delivery_mismatch'] = df['is_delivery']     * (1.0 - df['card_match'])
    df['card_pay_delivery'] = df['is_card_payment'] * df['is_delivery']
    return df

def prepare(df):
    df = df.drop(columns=DROP_COLS).copy()
    df[NUM_COLS] = df[NUM_COLS].astype(float)
    df[TARGET] = df[TARGET].astype(int)
    return df

dataset1 = prepare(dataset1)
dataset2 = prepare(dataset2)
dataset1 = add_interactions(dataset1)
dataset2 = add_interactions(dataset2)

for name, df in [("RTF_2", dataset1), ("RTF_1", dataset2)]:
    null_counts = df[ALL_FEATURE_COLS].isnull().sum()
    if null_counts.any():
        raise ValueError(f"{name}: null values in feature columns:\n{null_counts[null_counts > 0]}")
    print(f"{name}: {len(df)} rows, {len(ALL_FEATURE_COLS)} features "
          f"({len(NUM_COLS)} base + {len(INTERACTION_COLS)} interaction), no nulls")

RTF_2: 1201 rows, 11 features (8 base + 3 interaction), no nulls
RTF_1: 1199 rows, 11 features (8 base + 3 interaction), no nulls


## Train / val split

No timestamp column is available, so we use a stratified random split.
Pre-normalisation feature values are saved here so the review table shows human-readable originals.

In [8]:
train_df, val_df = train_test_split(
    dataset1, test_size=0.2, random_state=SEED, stratify=dataset1[TARGET]
)
test_df = dataset2.copy()

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Save pre-normalisation NUM_COLS only for the human-readable review table
val_order_ids  = val_df['order_id'].values
val_raw_feats  = val_df[NUM_COLS].copy()
test_order_ids = test_df['order_id'].values
test_raw_feats = test_df[NUM_COLS].copy()

# 25% of train held out for isotonic calibration
train_core_df, cal_df = train_test_split(
    train_df, test_size=0.25, random_state=SEED, stratify=train_df[TARGET]
)
train_core_df = train_core_df.reset_index(drop=True)
cal_df        = cal_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print(f"  train split: {len(train_core_df)} core + {len(cal_df)} calibration")

Train: 960  Val: 241  Test: 1199
  train split: 720 core + 240 calibration


## Normalisation

Statistics are fit on `train_core_df` only and applied to all four splits (core, cal, val, test).
`cal_df` is kept separate so the isotonic calibrator is fitted on genuinely held-out scores.

In [9]:
means = train_core_df[ALL_FEATURE_COLS].mean()
stds  = train_core_df[ALL_FEATURE_COLS].std().replace(0, 1)
for part in (train_core_df, cal_df, val_df, test_df):
    part.loc[:, ALL_FEATURE_COLS] = (part[ALL_FEATURE_COLS] - means) / stds

X_train = train_core_df[ALL_FEATURE_COLS].to_numpy(dtype="float32")
y_train = train_core_df[TARGET].to_numpy(dtype="int32")
X_cal   = cal_df[ALL_FEATURE_COLS].to_numpy(dtype="float32")
y_cal   = cal_df[TARGET].to_numpy(dtype="int32")
X_val   = val_df[ALL_FEATURE_COLS].to_numpy(dtype="float32")
y_val   = val_df[TARGET].to_numpy(dtype="int32")
X_test  = test_df[ALL_FEATURE_COLS].to_numpy(dtype="float32")
y_test  = test_df[TARGET].to_numpy(dtype="int32")

print(f"Train: {X_train.shape}  Cal: {X_cal.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

Train: (720, 11)  Cal: (240, 11)  Val: (241, 11)  Test: (1199, 11)


In [10]:
n_pos = int(train_core_df[TARGET].sum())
n_neg = len(train_core_df) - n_pos
scale_pos_weight = n_neg / n_pos
print(f"positives={n_pos}  negatives={n_neg}  fraud rate={n_pos/len(train_core_df):.4%}")
print(f"scale_pos_weight={scale_pos_weight:.2f}")

positives=120  negatives=600  fraud rate=16.6667%
scale_pos_weight=5.00


## Model training

In [11]:
model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr", early_stopping_rounds=20,
    random_state=SEED, n_jobs=-1,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

[0]	validation_0-aucpr:0.57971
[19]	validation_0-aucpr:0.49591


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,20
,enable_categorical,False
,eval_metric,'aucpr'


In [12]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, class_weight="balanced",
    random_state=SEED, n_jobs=-1,
)
rf.fit(X_train, y_train)
print("RandomForest trained")

lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
lr.fit(X_train, y_train)
print("LogisticRegression trained")

gnb = GaussianNB()
gnb.fit(X_train, y_train)
print("GaussianNB trained")

RandomForest trained
LogisticRegression trained
GaussianNB trained


In [13]:
# Train on legit transactions only -- learns what "normal" looks like
iso = IsolationForest(n_estimators=200, contamination="auto", random_state=SEED, n_jobs=-1)
iso.fit(X_train[y_train == 0])
print(f"IsolationForest trained on {(y_train == 0).sum():,} legit transactions")

val_if_scores  = -iso.score_samples(X_val)
test_if_scores = -iso.score_samples(X_test)

IsolationForest trained on 600 legit transactions


## Scoring

In [14]:
def report_auc(name, y, scores):
    pr  = average_precision_score(y, scores)
    roc = roc_auc_score(y, scores)
    print(f"  {name:22s}  PR-AUC: {pr:.4f}  ROC-AUC: {roc:.4f}")

xgb_val  = model.predict_proba(X_val)[:, 1]
rf_val   = rf.predict_proba(X_val)[:, 1]
lr_val   = lr.predict_proba(X_val)[:, 1]
gnb_val  = gnb.predict_proba(X_val)[:, 1]
xgb_test = model.predict_proba(X_test)[:, 1]
rf_test  = rf.predict_proba(X_test)[:, 1]
lr_test  = lr.predict_proba(X_test)[:, 1]
gnb_test = gnb.predict_proba(X_test)[:, 1]

print("--- Per-model validation AUC ---")
for nm, scores in [
    ("XGBoost", xgb_val), ("RandomForest", rf_val), ("LogisticReg", lr_val),
    ("GaussianNB", gnb_val), ("IsolationForest", val_if_scores),
]:
    report_auc(nm, y_val, scores)

val_ensemble  = (xgb_val + rf_val + lr_val) / 3
test_ensemble = (xgb_test + rf_test + lr_test) / 3
report_auc("Ensemble (raw)", y_val, val_ensemble)

# --- Probability calibration ---
# Isotonic regression on held-out cal_df reshapes scores toward 0/1, shrinking the review zone
xgb_cal_s = model.predict_proba(X_cal)[:, 1]
rf_cal_s  = rf.predict_proba(X_cal)[:, 1]
lr_cal_s  = lr.predict_proba(X_cal)[:, 1]
cal_raw   = (xgb_cal_s + rf_cal_s + lr_cal_s) / 3

calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(cal_raw, y_cal)

val_ensemble_cal  = calibrator.transform(val_ensemble)
test_ensemble_cal = calibrator.transform(test_ensemble)
report_auc("Ensemble (calibrated)", y_val, val_ensemble_cal)

--- Per-model validation AUC ---
  XGBoost                 PR-AUC: 0.5797  ROC-AUC: 0.9279
  RandomForest            PR-AUC: 0.5384  ROC-AUC: 0.9199
  LogisticReg             PR-AUC: 0.6041  ROC-AUC: 0.9308
  GaussianNB              PR-AUC: 0.5797  ROC-AUC: 0.9279
  IsolationForest         PR-AUC: 0.3690  ROC-AUC: 0.8372
  Ensemble (raw)          PR-AUC: 0.5297  ROC-AUC: 0.9205
  Ensemble (calibrated)   PR-AUC: 0.5523  ROC-AUC: 0.9192


## Three-tier threshold selection

Thresholds are selected on calibrated scores from the validation set.
The comparison below shows how calibration affects the review queue size vs raw ensemble scores.

- **`fraud_threshold`**: highest recall achievable at >=70% precision
- **`legit_threshold`**: highest threshold where recall is still >=90% (at most 10% of fraud slips into auto-legit)

In [15]:
MIN_PRECISION   = 0.55
MIN_RECALL_SAFE = 0.90

def select_thresholds(scores, labels, min_prec=MIN_PRECISION, min_rec=MIN_RECALL_SAFE):
    prec_c, rec_c, thr_c = precision_recall_curve(labels, scores)
    mask_p = prec_c[:-1] >= min_prec
    if mask_p.any():
        fi = int(np.argmax(rec_c[:-1] * mask_p.astype(float)))
        fraud_thr = float(thr_c[fi])
    else:
        fi = len(thr_c) - 1
        fraud_thr = float(thr_c[fi])
        print(f"WARNING: {min_prec:.0%} precision not achievable; fraud_threshold={fraud_thr:.4f}")
    mask_r = rec_c[:-1] >= min_rec
    if mask_r.any():
        li = int(np.where(mask_r)[0][-1])
        legit_thr = float(thr_c[li])
    else:
        legit_thr = 0.0
        print(f"WARNING: {min_rec:.0%} recall not achievable; legit_threshold=0.0")
    if legit_thr >= fraud_thr:
        legit_thr = 0.0
    return legit_thr, fraud_thr

def classify(scores, legit_thr, fraud_thr):
    if legit_thr == 0.0:
        # No uncertain mid-range: skip review tier, everything below fraud threshold is auto_legit
        return np.where(scores >= fraud_thr, "auto_fraud", "auto_legit")
    return np.where(
        scores >= fraud_thr, "auto_fraud",
        np.where(scores >= legit_thr, "review", "auto_legit"),
    )

def print_tier_breakdown(label, scores, labels, legit_thr, fraud_thr):
    decisions = classify(scores, legit_thr, fraud_thr)
    print(f"\n{label}  (fraud_thr={fraud_thr:.4f}  legit_thr={legit_thr:.4f})")
    for dec in ["auto_fraud", "review", "auto_legit"]:
        sub = labels[decisions == dec]
        n, n_fraud = len(sub), int(sub.sum())
        pct  = n / len(labels) * 100
        prec = n_fraud / n if n > 0 else 0.0
        print(f"  {dec:12s}: {n:3d} ({pct:4.1f}%)  -- {n_fraud} fraud, {n - n_fraud} legit  precision={prec:.0%}")
    return decisions

print("--- Threshold comparison: raw vs calibrated ---")
lt_raw, ft_raw = select_thresholds(val_ensemble, y_val)
_              = print_tier_breakdown("Raw ensemble", val_ensemble, y_val, lt_raw, ft_raw)

lt_cal, ft_cal = select_thresholds(val_ensemble_cal, y_val)
val_decisions  = print_tier_breakdown("Calibrated ensemble", val_ensemble_cal, y_val, lt_cal, ft_cal)
test_decisions = classify(test_ensemble_cal, lt_cal, ft_cal)

legit_threshold = lt_cal
fraud_threshold = ft_cal

print(f"\nFinal zones:")
if legit_threshold == 0.0:
    print(f"  score < {fraud_threshold:.4f}   -> auto-legit  (no uncertain mid-range; review tier suppressed)")
    print(f"  score >= {fraud_threshold:.4f}  -> auto-fraud")
else:
    print(f"  score < {legit_threshold:.4f}                   -> auto-legit")
    print(f"  {legit_threshold:.4f} <= score < {fraud_threshold:.4f}  -> manual review")
    print(f"  score >= {fraud_threshold:.4f}                  -> auto-fraud")

--- Threshold comparison: raw vs calibrated ---

Raw ensemble  (fraud_thr=0.1607  legit_thr=0.0000)
  auto_fraud  :  72 (29.9%)  -- 40 fraud, 32 legit  precision=56%
  review      :   0 ( 0.0%)  -- 0 fraud, 0 legit  precision=0%
  auto_legit  : 169 (70.1%)  -- 0 fraud, 169 legit  precision=0%

Calibrated ensemble  (fraud_thr=0.6571  legit_thr=0.0000)
  auto_fraud  :  69 (28.6%)  -- 40 fraud, 29 legit  precision=58%
  review      :   0 ( 0.0%)  -- 0 fraud, 0 legit  precision=0%
  auto_legit  : 172 (71.4%)  -- 0 fraud, 172 legit  precision=0%

Final zones:
  score < 0.6571   -> auto-legit  (no uncertain mid-range; review tier suppressed)
  score >= 0.6571  -> auto-fraud


## Validation set — tier breakdown

In [16]:
val_table = val_raw_feats.copy()
val_table.insert(0, 'order_id',    val_order_ids)
val_table['fraud_score']  = val_ensemble_cal
val_table['decision']     = val_decisions
val_table['actual_fraud'] = y_val

print("--- Validation set breakdown (calibrated + hard rules) ---")
for dec in ["auto_fraud", "review", "auto_legit"]:
    sub = val_table[val_table['decision'] == dec]
    n_fraud = int((sub['actual_fraud'] == 1).sum())
    n_legit = int((sub['actual_fraud'] == 0).sum())
    pct = len(sub) / len(val_table) * 100
    prec_str = f"  precision={n_fraud/len(sub):.0%}" if len(sub) > 0 else ""
    print(f"  {dec:12s}: {len(sub):3d} ({pct:4.1f}%)  -- {n_fraud} fraud, {n_legit} legit{prec_str}")

--- Validation set breakdown (calibrated + hard rules) ---
  auto_fraud  :  69 (28.6%)  -- 40 fraud, 29 legit  precision=58%
  review      :   0 ( 0.0%)  -- 0 fraud, 0 legit
  auto_legit  : 172 (71.4%)  -- 0 fraud, 172 legit  precision=0%


## Manual review queue

In [17]:
review_df = (
    val_table[val_table['decision'] == 'review']
    .sort_values('fraud_score', ascending=False)
    .reset_index(drop=True)
)

display_cols = ['order_id', 'fraud_score', 'card_match', 'cvv'] + BOOL_COLS + ['actual_fraud']

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.3f}'.format)

print(f"MANUAL REVIEW QUEUE  ({len(review_df)} transactions, sorted by fraud score)")
print("-" * 60)

if len(review_df) > 0:
    disp = review_df[display_cols].copy()
    for col in BOOL_COLS:
        disp[col] = disp[col].astype(int)
    disp['card_match'] = disp['card_match'].astype(int)
    disp['cvv'] = disp['cvv'].astype(int)
    display(disp)
else:
    print("  (no transactions in review zone)")

# Save for analyst labeling -- add 'is_fraud' column, merge into RTF_2 before next retraining
review_df[['order_id', 'fraud_score'] + NUM_COLS].to_csv(
    "../00_reference/review_queue.csv", index=False
)
print("\nReview queue saved to ../00_reference/review_queue.csv")
print("  -> Label the 'is_fraud' column and merge into training data for next retraining cycle.")

MANUAL REVIEW QUEUE  (0 transactions, sorted by fraud score)
------------------------------------------------------------
  (no transactions in review zone)

Review queue saved to ../00_reference/review_queue.csv
  -> Label the 'is_fraud' column and merge into training data for next retraining cycle.


## Test set — out-of-distribution evaluation (RTF_1)

In [18]:
test_table = test_raw_feats.copy()
test_table.insert(0, 'order_id',    test_order_ids)
test_table['fraud_score']  = test_ensemble_cal
test_table['decision']     = test_decisions
test_table['actual_fraud'] = y_test

print("--- Test set (RTF_1, out-of-distribution) ---")
for dec in ["auto_fraud", "review", "auto_legit"]:
    sub = test_table[test_table['decision'] == dec]
    n_fraud = int((sub['actual_fraud'] == 1).sum())
    n_legit = int((sub['actual_fraud'] == 0).sum())
    pct = len(sub) / len(test_table) * 100
    prec_str = f"  precision={n_fraud/len(sub):.0%}" if len(sub) > 0 else ""
    print(f"  {dec:12s}: {len(sub):3d} ({pct:4.1f}%)  -- {n_fraud} fraud, {n_legit} legit{prec_str}")

print(f"\nTest PR-AUC: {average_precision_score(y_test, test_ensemble_cal):.4f}  "
      f"ROC-AUC: {roc_auc_score(y_test, test_ensemble_cal):.4f}")

--- Test set (RTF_1, out-of-distribution) ---
  auto_fraud  : 1027 (85.7%)  -- 999 fraud, 28 legit  precision=97%
  review      :   0 ( 0.0%)  -- 0 fraud, 0 legit
  auto_legit  : 172 (14.3%)  -- 1 fraud, 171 legit  precision=1%

Test PR-AUC: 0.9752  ROC-AUC: 0.9361


## Distribution shift monitor

Tracks whether the incoming predicted fraud rate has diverged from the training baseline.
A shift >20 ppts suggests the model should be retrained on more recent labeled data.

**Retraining cycle:**
1. Analysts label the `review_queue.csv` outputs (`is_fraud` column)
2. Merge confirmed labels back into RTF_2 (expanding the training set)
3. Retrain from this cell upward
4. Reselect thresholds on the new validation set

In [19]:
def check_drift(name, scores, fraud_thr, ref_rate, tol=0.20):
    predicted = (scores >= fraud_thr).mean()
    shift = abs(predicted - ref_rate)
    status = "DRIFT -- consider retraining" if shift > tol else "OK"
    print(f"  {name:35s}  predicted={predicted:.1%}  ref={ref_rate:.1%}  shift={shift:.1%}  [{status}]")

print(f"--- Distribution shift monitor (training baseline: {fraud_train:.1%} fraud) ---")
check_drift("Val set (RTF_2 holdout)",      val_ensemble_cal,  fraud_threshold, fraud_train)
check_drift("Test set (RTF_1, OOD)",        test_ensemble_cal, fraud_threshold, fraud_train)

--- Distribution shift monitor (training baseline: 16.7% fraud) ---
  Val set (RTF_2 holdout)              predicted=28.6%  ref=16.7%  shift=12.0%  [OK]
  Test set (RTF_1, OOD)                predicted=85.7%  ref=16.7%  shift=69.0%  [DRIFT -- consider retraining]
